# 08 · Explicabilidade (LIME) e equidade (Fairlearn)

LIME verifica se o modelo decide pelas palavras certas. Fairlearn compara o desempenho entre grupos: gênero, região e faixa etária do avaliador no RePro; região na Olist.

Ambiente: **GPU T4**.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[transformers,analise]"
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import numpy as np
import pandas as pd
from absa.config import ASPECTS, LABEL_COLUMNS, MODELS, POLARITIES
from absa.data import UF2REGIAO
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.repro import load_repro
treino, val, teste, _ = load_repro()
print(len(treino), len(val), len(teste))

MELHOR = "bertimbau_humano"
clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS[MELHOR.split("_")[0]])).load(MODELS / f"{MELHOR}.pt")
pred = clf.predict(teste.texto.map(clean_light).tolist())

## LIME: por que o modelo marcou "entrega negativa"?

In [ ]:
from lime.lime_text import LimeTextExplainer

aspecto = "entrega"
j = ASPECTS.index(aspecto)
explainer = LimeTextExplainer(class_names=POLARITIES, random_state=0)

def proba_aspecto(textos):
    return clf.predict_proba([clean_light(t) for t in textos])[:, j, :]

casos = teste[teste[f"asp_{aspecto}"] == "negativo"].sample(3, random_state=1)
for texto in casos.texto:
    exp = explainer.explain_instance(texto, proba_aspecto, labels=[POLARITIES.index("negativo")], num_features=8, num_samples=500)
    print(texto)
    print(exp.as_list(label=POLARITIES.index("negativo")), "\n")

## Fairlearn: o modelo erra mais para algum grupo?

In [ ]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import f1_score

grupos = pd.DataFrame({
    "genero": teste.genero.fillna("NI"),
    "regiao": teste.uf.map(UF2REGIAO).fillna("NI"),
    "idade": pd.cut(2018 - teste.ano_nascimento, [0, 30, 45, 60, 120], labels=["<30", "30-45", "45-60", "60+"]).astype(str),
})

def f1_macro(y, p):
    return f1_score(y, p, average="macro", zero_division=0)

for attr in grupos:
    linhas = {}
    for col in LABEL_COLUMNS:
        mf = MetricFrame(metrics=f1_macro, y_true=teste[col], y_pred=pred[col], sensitive_features=grupos[attr])
        linhas[col] = mf.by_group
    tab = pd.DataFrame(linhas).round(3)
    tab["n"] = grupos[attr].value_counts()
    print(f"\n== {attr} ==")
    display(tab)

Diferenças só merecem atenção se forem maiores que o intervalo de confiança do grupo: grupos pequenos (ex.: Norte) oscilam muito por acaso.